## Data Cleaning

Loading the dataset

In [10]:
import pandas as pd

df = pd.read_csv("../data/raw/Sample - Superstore.csv", encoding="latin1")

Standarizing column names

In [11]:
#to make code cleaner accross sql and powwer bi
#we standarize column names to lower case and replace spaces with underscores

#print orginal column names§
print("Original columns:")
print(df.columns.tolist())


Original columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [12]:
#rename the columns
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
)

In [13]:
#print them again
print("Renamed columns:")
print(df.columns.tolist())

Renamed columns:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


Converting data types

In [14]:
#inspecting the first 10 rows of the order_date column
df["order_date"].head(10)

0     11/8/2016
1     11/8/2016
2     6/12/2016
3    10/11/2015
4    10/11/2015
5      6/9/2014
6      6/9/2014
7      6/9/2014
8      6/9/2014
9      6/9/2014
Name: order_date, dtype: object

In [16]:
##inspecting the first 10 rows of the ship_date column
df["ship_date"].head(10)


0    11/11/2016
1    11/11/2016
2     6/16/2016
3    10/18/2015
4    10/18/2015
5     6/14/2014
6     6/14/2014
7     6/14/2014
8     6/14/2014
9     6/14/2014
Name: ship_date, dtype: object

In [22]:
date_columns = ["order_date", "ship_date"]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="%m/%d/%Y"
    )

Checking null values

In [ ]:
#in this dataset, there are no missing values.
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing

,Missing Values,Percentage
row_id,0,0.0
order_id,0,0.0
order_date,0,0.0
ship_date,0,0.0
ship_mode,0,0.0
customer_id,0,0.0
customer_name,0,0.0
segment,0,0.0
country,0,0.0
city,0,0.0


Checking for duplicates

In [ ]:
#in this dataset, there are no duplicate rows.
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 0


Removing invalid white spaces

In [ ]:
#Remove leading/trailing spaces from text columns
object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    df[col] = df[col].str.strip()

Checking inconsistent capitalization

In [ ]:
#in this dataset, capitalization is consistent.
for col in ["category", "sub_category", "segment", "region"]:
    print(f"\n{col}")
    print(df[col].unique())


category
['Furniture' 'Office Supplies' 'Technology']

sub_category
['Bookcases' 'Chairs' 'Labels' 'Tables' 'Storage' 'Furnishings' 'Art'
 'Phones' 'Binders' 'Appliances' 'Paper' 'Accessories' 'Envelopes'
 'Fasteners' 'Supplies' 'Machines' 'Copiers']

segment
['Consumer' 'Corporate' 'Home Office']

region
['South' 'West' 'Central' 'East']


Checking for impossible numeric values

In [ ]:
df[["sales", "quantity", "discount", "profit"]].describe()

,sales,quantity,discount,profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [ ]:
(df["sales"] < 0).sum()


0

In [30]:
(df["quantity"] <= 0).sum()

0

In [31]:
((df["discount"] < 0) | (df["discount"] > 1)).sum()

0

Validate business rules

In [32]:
#Shipping date should never be before order date.
(df["ship_date"] < df["order_date"]).sum()

0

Checking for invalid categorical values

In [33]:
categorical_columns = [
    "category",
    "sub_category",
    "segment",
    "region",
    "ship_mode"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())


category
category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

sub_category
sub_category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64

segment
segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64

region
region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64

ship_mode
ship_mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64


Checking for outliers

In [34]:
df[["sales", "profit"]].describe(percentiles=[0.25,0.5,0.75,0.95,0.99])

,sales,profit
count,9994.000000,9994.000000
mean,229.858001,28.656896
std,623.245101,234.260108
min,0.444000,-6599.978000
25%,17.280000,1.728750
50%,54.490000,8.666500
75%,209.940000,29.364000
95%,956.984245,168.470400
99%,2481.694600,580.657882
max,22638.480000,8399.976000


Final quality check

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   row_id         9994 non-null   int64         
 1   order_id       9994 non-null   object        
 2   order_date     9994 non-null   datetime64[ns]
 3   ship_date      9994 non-null   datetime64[ns]
 4   ship_mode      9994 non-null   object        
 5   customer_id    9994 non-null   object        
 6   customer_name  9994 non-null   object        
 7   segment        9994 non-null   object        
 8   country        9994 non-null   object        
 9   city           9994 non-null   object        
 10  state          9994 non-null   object        
 11  postal_code    9994 non-null   int64         
 12  region         9994 non-null   object        
 13  product_id     9994 non-null   object        
 14  category       9994 non-null   object        
 15  sub_category   9994 n

Export cleaned dataset

In [36]:
df.to_csv(
    "../data/processed/clean_superstore.csv",
    index=False
)

| Task                         | Result                                               |
| ---------------------------- | ---------------------------------------------------- |
| Loaded dataset               | ✅                                                   |
| Standardized column names    | ✅                                                   |
| Converted date columns       | ✅                                                   |
| Checked missing values       | No missing values found                              |
| Checked duplicate rows       | No duplicates found.                                 |
| Removed extra whitespace     | ✅                                                   |
| Validated categorical values | No inconsistencies found                             |
| Validated numeric values     | All business rules satisfied                         |
| Verified shipping dates      | No invalid records                                   |
| Exported cleaned dataset     | `clean_superstore.csv`                               |
